# 🧹 Data Cleaning Pipeline — Sales Dataset

**Компания:** ShopSphere (гипотетический датасет продаж)  
**Цель:** Очистить и стандартизировать сырые данные продаж для дальнейшего использования в аналитике и отчётности  
**Исходный файл:** `sales_data_messy.csv`  
**Результат:** `sales_data_clean.csv`

---

## Содержание
1. Загрузка данных и первичный профайлинг
2. Стандартизация текстовых данных
3. Исправление типов данных
4. Обработка пропущенных значений
5. Удаление дубликатов и генерация уникального ID
6. Обработка выбросов
7. Проверка логической консистентности
8. Экспорт чистого датасета
9. Итоговый отчёт

## 1. Загрузка данных и первичный профайлинг

Загружаем датасет и проводим первичный осмотр — структура, типы данных, пропуски, уникальные значения.

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("sales_data_messy.csv")
print(df.shape)
df.head(10)

(155, 10)


,TransactionID,OrderDate,Region,ProductCategory,ProductName,Quantity,UnitPrice,TotalPrice,CustomerID,CustomerSegment
0,trn_0014,2022-10-02,Central,Books,Novel - Fiction,8.0,726.570260,5812.56,5000,Enterprise
1,TRN-0089,2022-05-01,Western,Software,Smartphone X,7.0,526.440480,3685.08,3502,SME
2,TRN-0065,2022-10-29,East,electronics,AV Suite,18.0,135.775282,2443.96,1830,Large Business
3,TRN-0007,2022-06-23,NORTH,Apparel,Novel - Fiction,9.0,651.853225,5866.68,5752,SME
4,TRN-0107,2022-03-21,East,Electronics,Laptop Pro,17.0,415.842842,7069.33,5079,Consumer
5,TRN-0029,NaN,Missing,SW,AV Suite,NaN,651.446384,NaN,NaN,Large Business
6,TRN-0029,2022-01-26,NORTH,Electronics,Desk Chair Ergo,2.0,974.360027,1948.72,6726,SMB
7,TRN-0027,10/24/2022,Northern,SW,Laptop Pro v2.0,1.0,895.505075,895.51,9335,UNKNOWN
8,trn_0027,2023-01-01,Midwest,Home & Garden,Fiction Novel,12.0,314.714141,3776.57,6222,missing
9,trn_0029,2022-08-28,E. Region,Elec.,Ergonomic Chair,13.0,129.932967,1858.04,3849,SMB


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   TransactionID    155 non-null    object 
 1   OrderDate        140 non-null    object 
 2   Region           150 non-null    object 
 3   ProductCategory  155 non-null    object 
 4   ProductName      155 non-null    object 
 5   Quantity         151 non-null    float64
 6   UnitPrice        147 non-null    float64
 7   TotalPrice       144 non-null    float64
 8   CustomerID       135 non-null    object 
 9   CustomerSegment  155 non-null    object 
dtypes: float64(3), object(7)
memory usage: 12.2+ KB


In [3]:
df.describe()

,Quantity,UnitPrice,TotalPrice
count,151.000000,147.000000,144.000000
mean,12.350993,661.687559,9306.155069
std,14.975624,777.676810,24203.484335
min,1.000000,0.830000,2.660000
25%,5.000000,286.206591,1482.825000
50%,10.000000,527.545072,4049.030000
75%,14.000000,819.250411,9382.650000
max,97.000000,4892.080000,257442.570000


In [4]:
print('Пропуски по колонкам:')
print(df.isna().sum())

Пропуски по колонкам:
TransactionID       0
OrderDate          15
Region              5
ProductCategory     0
ProductName         0
Quantity            4
UnitPrice           8
TotalPrice         11
CustomerID         20
CustomerSegment     0
dtype: int64


**Находки профайлинга:**
- Всего строк: 155, колонок: 10
- Пропуски в 6 колонках: OrderDate (15), Region (5), Quantity (4), UnitPrice (8), TotalPrice (11), CustomerID (20)
- Текстовые колонки содержат дубликаты написания (ProductName, Region, CustomerSegment, ProductCategory)
- OrderDate представлен в двух разных форматах (MM/DD/YYYY и DD-MM-YYYY)
- TotalPrice содержит 22 несоответствия формуле Quantity × UnitPrice

---
## 2. Стандартизация текстовых данных

Унифицируем написание значений в текстовых колонках. Разные написания одного и того же значения (например 'Laptop Pro', 'LaptopPro', 'Laptop-Pro') приводят к неправильной группировке при анализе.

### 2.1 ProductName

**Найденные варианты и решения:**
- `Laptop Pro v2.0`, `LaptopPro`, `Laptop-Pro` → `Laptop Pro` (одна и та же модель в разных форматах)
- `Smart Phone X` → `Smartphone X` (пробел внутри слова)
- `AV Suite` → `Anti-virus Suite` (аббревиатура vs полное название)
- `Novel - Fiction` → `Fiction Novel` (перестановка слов)
- `Desk Chair Ergo` → `Ergonomic Chair` (сокращённое vs полное название)

**Допущение:** Принято что каждая из групп выше — один и тот же товар. При наличии продуктового каталога решение должно быть согласовано с командой.

In [5]:
product_mapping = {"Laptop Pro v2.0": "Laptop Pro",
                   "LaptopPro": "Laptop Pro",
                   "Laptop-Pro": "Laptop Pro",
                   "Smart Phone X": "Smartphone X",
                   "AV Suite": "Anti-virus Suite",
                   "Novel - Fiction": "Fiction Novel", 
                   "Desk Chair Ergo": "Ergonomic Chair"}
df["ProductName"] = df["ProductName"].replace(product_mapping)
print(df['ProductName'].unique())

['Fiction Novel' 'Smartphone X' 'Anti-virus Suite' 'Laptop Pro'
 'Ergonomic Chair']


### 2.2 Region

**Найденные варианты и решения:**
- `Western` → `West`
- `E. Region` → `East`
- `NORTH`, `Northern`, `N. Region` → `North`
- `SOUTHERN`, `S. Region` → `South`
- `Missing`, `NaN` → `Unknown`

**Допущение:** Значение `Missing` интерпретировано как отсутствующие данные и заменено на `Unknown` наравне с NaN.

In [6]:
region_mapping = {"Central": "Central",
                  "Western": "West",
                  "E. Region": "East",
                  "NORTH": "North",
                  "Northern": "North",
                  "N. Region": "North",
                  "SOUTHERN": "South",
                  "S. Region": "South",
                  "Missing": np.nan,
                  None: np.nan}
df['Region'] = df['Region'].replace(region_mapping)
df['Region'] = df['Region'].fillna('Unknown')
print(df['Region'].unique())

['Central' 'West' 'East' 'North' 'Unknown' 'Midwest' 'South']


### 2.3 CustomerSegment

**Найденные варианты и решения:**
- `Large Business`, `Corp` → `Enterprise`
- `SME`, `Small Business` → `SMB`
- `Consumer` → `Individual`
- `UNKNOWN`, `missing` → `Unknown`

**Допущение:** SME (Small & Medium Enterprise) приравнено к SMB (Small & Medium Business) как эквивалентные термины.

In [7]:
customer_segment_mapping = {"Large Business": "Enterprise",
                            "Corp": "Enterprise",
                            "SME": "SMB",
                            "Small Business": "SMB",
                            "Consumer": "Individual",
                            "UNKNOWN": "Unknown",
                            "missing": "Unknown"}
df['CustomerSegment'] = df['CustomerSegment'].replace(customer_segment_mapping)
print(df['CustomerSegment'].unique())

['Enterprise' 'SMB' 'Individual' 'Unknown']


### 2.4 ProductCategory

**Найденные варианты и решения:**
- `SW` → `Software`
- `Clothing` → `Apparel`
- `Elec.`, `electronics` → `Electronics`

**Допущение:** `Clothing` и `Apparel` считаются одной категорией. При наличии официального каталога категорий решение должно быть согласовано.

In [9]:
product_category_mapping = {"electronics": "Electronics",
                            "Elec.": "Electronics",
                            "Home Goods": "Home & Garden",
                            "SW": "Software",
                            "Clothing": "Apparel"}
df['ProductCategory'] = df['ProductCategory'].replace(product_category_mapping)
print(df['ProductCategory'].unique())

['Books' 'Software' 'Electronics' 'Apparel' 'Home & Garden'
 'Printed Media']


---
## 3. Исправление типов данных

Колонка `OrderDate` загружается как `object` (текст). Необходимо привести к типу `datetime64` для корректной работы с датами.

**Проблема:** В колонке присутствуют два формата дат:
- `MM/DD/YYYY` (американский формат, например `10/24/2022`)
- `DD-MM-YYYY` (европейский формат, например `26-07-2022`)
- `YYYY-MM-DD` (ISO формат, например `2022-10-02`)

**Решение:** Обработать каждый формат отдельно через маски.

In [10]:

mask_slash = df['OrderDate'].str.contains('/', na=False)
mask_dash = df['OrderDate'].str.contains('-', na=False) & ~df['OrderDate'].str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
mask_iso = df['OrderDate'].str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)

dates_slash = pd.to_datetime(df.loc[mask_slash, 'OrderDate'], format='%m/%d/%Y', errors='coerce')
dates_dash = pd.to_datetime(df.loc[mask_dash, 'OrderDate'], format='%d-%m-%Y', errors='coerce')
dates_iso = pd.to_datetime(df.loc[mask_iso, 'OrderDate'], format='%Y-%m-%d', errors='coerce')

df['OrderDate'] = pd.concat([dates_slash, dates_dash, dates_iso]).sort_index()
print(df['OrderDate'].dtype)
print(f'Пропусков в OrderDate: {df["OrderDate"].isna().sum()}')

datetime64[ns]
Пропусков в OrderDate: 15


**Результат:** 15 строк остались с NaT (Not a Time) — это оригинальные пропуски из исходного датасета, дату для которых невозможно восстановить.

---
## 4. Обработка пропущенных значений

Стратегия для каждой колонки выбирается индивидуально в зависимости от природы данных.

### 4.1 Quantity и UnitPrice — восстановление через формулу

Если два из трёх связанных полей (`Quantity`, `UnitPrice`, `TotalPrice`) заполнены — третье можно вычислить.

**Допущение:** `TotalPrice = Quantity × UnitPrice` — эта формула принята как основная.

In [11]:

mask_q = df['Quantity'].isna() & df['TotalPrice'].notna() & df['UnitPrice'].notna()
df.loc[mask_q, 'Quantity'] = (df.loc[mask_q, 'TotalPrice'] / df.loc[mask_q, 'UnitPrice']).round()

mask_p = df['UnitPrice'].isna() & df['TotalPrice'].notna() & df['Quantity'].notna()
df.loc[mask_p, 'UnitPrice'] = (df.loc[mask_p, 'TotalPrice'] / df.loc[mask_p, 'Quantity']).round(2)

print(f'Пропусков Quantity: {df["Quantity"].isna().sum()}')
print(f'Пропусков UnitPrice: {df["UnitPrice"].isna().sum()}')

Пропусков Quantity: 4
Пропусков UnitPrice: 8


### 4.2 Оставшиеся пропуски — заполнение медианой по товару

Для строк где невозможно восстановить значение через формулу — заполняем медианой в разрезе `ProductName`.

**Допущение:** Медиана по товару точнее общей медианы — у разных товаров разные диапазоны цен и количеств.

In [12]:

df['Quantity'] = df['Quantity'].fillna(df.groupby('ProductName')['Quantity'].transform('median'))
df['UnitPrice'] = df['UnitPrice'].fillna(df.groupby('ProductName')['UnitPrice'].transform('median'))

print(f'Пропусков Quantity: {df["Quantity"].isna().sum()}')
print(f'Пропусков UnitPrice: {df["UnitPrice"].isna().sum()}')

Пропусков Quantity: 0
Пропусков UnitPrice: 0


### 4.3 CustomerID — гостевые покупки

20 пропусков в `CustomerID` интерпретированы как покупки без регистрации аккаунта.

**Допущение:** Пропуск CustomerID = гостевая покупка. Заполнено значением `GUEST`.

In [13]:
df['CustomerID'] = df['CustomerID'].fillna('GUEST')
print(f'Пропусков CustomerID: {df["CustomerID"].isna().sum()}')

Пропусков CustomerID: 0


### 4.4 OrderDate — оставляем NaT

15 пропусков в `OrderDate` оставлены как `NaT` — дату невозможно восстановить без дополнительных источников данных.

---
## 5. Удаление дубликатов и генерация уникального ID

### 5.1 Полные дубликаты строк

In [14]:
print(f'Полных дубликатов: {df.duplicated().sum()}')
df = df.drop_duplicates()
print(f'Строк после удаления: {len(df)}')

Полных дубликатов: 5
Строк после удаления: 150


### 5.2 Генерация уникального TransactionID

Обнаружено 12 коллизий в `TransactionID` — разные транзакции имели одинаковый ID. Это означает что исходная система генерации ID некорректна.

**Решение:** Перегенерировать все TransactionID как последовательные уникальные идентификаторы.

**Допущение:** Исходные TransactionID не несут смысловой нагрузки и не связаны с другими системами — поэтому безопасно заменить.

In [15]:
df = df.reset_index(drop=True)
df['TransactionID'] = 'TRN-' + (df.index + 1).astype(str).str.zfill(5)
print(f'Дубликатов TransactionID: {df["TransactionID"].duplicated().sum()}')
print(df['TransactionID'].head())

Дубликатов TransactionID: 0
0    TRN-00001
1    TRN-00002
2    TRN-00003
3    TRN-00004
4    TRN-00005
Name: TransactionID, dtype: object


---
## 6. Обработка выбросов

Используем метод IQR (Interquartile Range) для выявления выбросов.
Стратегия: **capping** (ограничение) — заменяем выбросы на граничное значение, не удаляем строки.

**Обоснование:** Удаление строк с выбросами может привести к потере реальных транзакций (например крупный оптовый заказ). Capping сохраняет строки, но ограничивает влияние экстремальных значений на анализ.

### 6.1 Quantity

In [16]:
Q1 = df['Quantity'].quantile(0.25)
Q3 = df['Quantity'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
outliers_q = df[df['Quantity'] > upper_bound]
print(f'Выбросов в Quantity: {len(outliers_q)}')
print(f'Верхняя граница: {upper_bound}')

df['Quantity'] = df['Quantity'].clip(upper=upper_bound)

Выбросов в Quantity: 6
Верхняя граница: 27.5


### 6.2 UnitPrice

In [17]:

Q1_p = df['UnitPrice'].quantile(0.25)
Q3_p = df['UnitPrice'].quantile(0.75)
IQR_p = Q3_p - Q1_p
upper_p = Q3_p + 1.5 * IQR_p

outliers_p = df[(df['UnitPrice'] < 0) | (df['UnitPrice'] > upper_p)]
print(f'Выбросов в UnitPrice: {len(outliers_p)}')
print(f'Верхняя граница: {upper_p:.2f}')

df['UnitPrice'] = df['UnitPrice'].clip(lower=0, upper=upper_p)

Выбросов в UnitPrice: 7
Верхняя граница: 1571.46


---
## 7. Проверка логической консистентности

Пересчитываем `TotalPrice = Quantity × UnitPrice` после всех предыдущих операций.

**Обоснование:** `Quantity` и `UnitPrice` считаются первичными полями ввода. `TotalPrice` должен всегда соответствовать их произведению.

In [18]:
df['TotalPrice'] = (df['Quantity'] * df['UnitPrice']).round(2)

mismatch = (df['TotalPrice'] - df['Quantity'] * df['UnitPrice']).abs() > 0.01
print(f'Несоответствий TotalPrice: {mismatch.sum()}')

Несоответствий TotalPrice: 0


---
## 8. Экспорт чистого датасета

In [19]:
df.to_csv('sales_data_clean.csv', index=False)
print(f'Сохранено: {len(df)} строк, {len(df.columns)} колонок')
print(df.isna().sum())

Сохранено: 150 строк, 10 колонок
TransactionID       0
OrderDate          15
Region              0
ProductCategory     0
ProductName         0
Quantity            0
UnitPrice           0
TotalPrice          0
CustomerID          0
CustomerSegment     0
dtype: int64


---
## 9. Итоговый отчёт

### Сводка выполненных операций

| Шаг | Операция | Результат |
|-----|----------|-----------|
| 1 | Загрузка и профайлинг | 155 строк, 10 колонок, выявлены проблемы |
| 2 | Стандартизация текста | ProductName: 12→5, Region: 14→6, CustomerSegment: 10→4, ProductCategory: 8→6 |
| 3 | Исправление типов | OrderDate: object→datetime64, 2 формата дат обработаны |
| 4 | Пропуски | Quantity/UnitPrice: восстановлены через формулу+медиана, CustomerID: GUEST, OrderDate: NaT |
| 5 | Дубликаты | Удалено 5 строк, TransactionID перегенерирован (12 коллизий) |
| 6 | Выбросы | Quantity: capping на 27.5, UnitPrice: capping на 1571.46 |
| 7 | Консистентность | TotalPrice пересчитан по формуле, 22 несоответствия исправлены |
| 8 | Экспорт | sales_data_clean.csv, 150 строк |

### Ключевые допущения
1. Группировки вариантов написания товаров/регионов/сегментов основаны на смысловой близости — при наличии официального справочника решения должны быть согласованы с владельцем данных
2. `Missing` в Region интерпретирован как пропущенное значение, а не отдельная категория
3. Пропуски в CustomerID = гостевые покупки без аккаунта
4. `TotalPrice` пересчитан по формуле — `UnitPrice` и `Quantity` считаются первичными полями
5. Capping выбросов вместо удаления — для сохранения реальных транзакций
6. 15 пропусков OrderDate оставлены как NaT — дата не восстанавливаема